# 04 CLIP Features (Cross-Modal)

CLIP is a pretrained model that maps images and text into the same vector space, so a photo and a caption describing the same thing end up close together. We use it **frozen** (no training) to measure whether each caption fits its image, the signal needed for `text_swap`, where a real caption from another article about the same person is attached to the photo.

**Saved feature groups** (`data/features/<group>_<split>.parquet`, joined on `row_id`, same convention as `03`):

| Group | Contents |
|---|---|
| `clip` | Image-caption similarity and normalised variants (a few columns) |
| `clip_emb` | The raw CLIP image and caption embeddings (768 + 768 columns, float16) |
| `clip_face` | Similarity between each face crop and the caption, summarised across faces |

**Model choice.** On 3,000 validation pairs, image-caption similarity alone separated `text_swap` from untouched captions with AUC 0.880 using CLIP ViT-B/32 and 0.917 with ViT-L/14. The larger model is about 6x slower but clearly better, so we use **ViT-L/14** (`openai/clip-vit-large-patch14`).

**Runtime** on an M2 Max (MPS): captions take a few minutes, whole images about 45 minutes, face crops about 2 hours. Embeddings are cached in `data/interim/`, so re-running skips work already done.

## 0. Setup

In [ ]:
import os
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_auc_score

os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"

PROJECT_ROOT = Path("..").resolve()
sys.path.insert(0, str(PROJECT_ROOT))

from transformers.utils import logging as hf_logging
from src.clip_features import (CLIP_NAME, ImageDataset, best_device, cosine_rows, encode_images,
                               encode_texts, load_clip)

hf_logging.disable_progress_bar()

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
FEATURES_DIR = PROJECT_ROOT / "data" / "features"
CACHE_DIR = PROJECT_ROOT / "data" / "interim" / "clip"
FEATURES_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

SPLITS = ["train_sample", "validation", "test"]
RANDOM_STATE = 4012
NUM_WORKERS = max(1, (os.cpu_count() or 2) - 4)

sns.set_theme(style="whitegrid")

In [ ]:
data = {split: pd.read_parquet(PROCESSED_DIR / f"{split}.parquet") for split in SPLITS}

# Set CLIP_SMOKE_TEST=1 to run the whole notebook on a few rows per split (separate cache, nothing saved to data/features)
SMOKE_TEST = os.environ.get("CLIP_SMOKE_TEST") == "1"
if SMOKE_TEST:
    data = {split: df.head(300) for split, df in data.items()}
    CACHE_DIR = CACHE_DIR / "smoke"
    CACHE_DIR.mkdir(parents=True, exist_ok=True)
    FEATURES_DIR = CACHE_DIR
all_rows = pd.concat(data.values(), ignore_index=True)

captions = pd.Series(all_rows["text"].unique())
images = all_rows.drop_duplicates("image")[["image", "mtcnn_boxes"]].reset_index(drop=True)
print(f"{len(all_rows):,} rows | {len(captions):,} distinct captions | {len(images):,} distinct images | "
      f"{images['mtcnn_boxes'].str.len().sum():,} face crops")

device = best_device()
model, image_processor, tokenizer = load_clip(device)
print(f"{CLIP_NAME} on {device}, {NUM_WORKERS} loader workers")

In [ ]:
def save_features(features, group):
    """Save one feature group per split, aligned to each split's rows (same helper as 03)."""
    for split, df in data.items():
        split_features = df[["row_id"]].merge(features, on="row_id", how="left")
        assert len(split_features) == len(df), f"{group}: row count changed for {split}"
        assert split_features.drop(columns="row_id").notna().all().all(), f"{group}: missing values for {split}"
        path = FEATURES_DIR / f"{group}_{split}.parquet"
        split_features.to_parquet(path, index=False)
        print(f"{path.relative_to(PROJECT_ROOT)}  {split_features.shape}")


def cached(name, compute):
    """Load an embedding array from the cache, or compute and cache it."""
    path = CACHE_DIR / f"{name}.npy"
    if path.exists():
        print(f"{name}: loaded from cache")
        return np.load(path)
    start = time.time()
    result = compute()
    np.save(path, result)
    print(f"{name}: computed in {(time.time() - start) / 60:.1f} min")
    return result


def auc_by_manipulation(df, col, higher_means_fake=False):
    """Single-feature AUC on train_sample: each manipulation type vs untouched, plus the two targets."""
    score = df[col] if higher_means_fake else -df[col]
    result = {}
    for target in ["image_fake", "text_fake"]:
        result[target] = roc_auc_score(df[target], score)
    for manipulation, column in [("face_swap", "image_manipulation"), ("face_attribute", "image_manipulation"),
                                 ("text_swap", "text_manipulation"), ("text_attribute", "text_manipulation")]:
        subset = df[column].isin(["none", manipulation])
        result[manipulation] = roc_auc_score(df.loc[subset, column] == manipulation, score[subset])
    return pd.Series(result, name=col)

## 1. Caption and Image Embeddings

In [ ]:
caption_emb = cached("captions", lambda: encode_texts(model, tokenizer, captions, device))
image_emb = cached("images", lambda: encode_images(model, ImageDataset(images["image"]), device,
                                                   num_workers=NUM_WORKERS))
print("caption embeddings:", caption_emb.shape, "| image embeddings:", image_emb.shape)

In [ ]:
# Map every row to its caption and image embedding
caption_index = pd.Series(np.arange(len(captions)), index=captions.values)
image_index = pd.Series(np.arange(len(images)), index=images["image"].values)
row_caption = caption_index.loc[all_rows["text"]].to_numpy()
row_image = image_index.loc[all_rows["image"]].to_numpy()

## 2. Image-Caption Similarity

- `clip_sim`: cosine similarity between the image and caption embeddings. Low means the caption does not fit the photo.
- Raw similarity also depends on how *specific* a caption or image is: a vague caption fits every photo moderately well, a detailed one fits only its own. So we also normalise against a fixed reference set of 2,000 images and 2,000 captions drawn from `train_sample`:
  - `clip_sim_txt_norm`: similarity minus the caption's average similarity to the reference images ("does this caption fit *this* photo better than photos in general?")
  - `clip_sim_img_norm`: similarity minus the image's average similarity to the reference captions
  - `clip_sim_txt_rank`: share of reference images this caption fits *worse* than its own image

The reference set comes from training data only and its labels are not used, so applying it to validation and test does not leak information.

In [ ]:
rng = np.random.default_rng(RANDOM_STATE)
train_mask = all_rows["row_id"].str.startswith("train_").to_numpy()
N_REF = 2000
train_images, train_captions = np.unique(row_image[train_mask]), np.unique(row_caption[train_mask])
ref_images = rng.choice(train_images, size=min(N_REF, len(train_images)), replace=False)
ref_captions = rng.choice(train_captions, size=min(N_REF, len(train_captions)), replace=False)

# Similarity of every distinct caption to the reference images, and every distinct image to the reference captions
caption_vs_ref = caption_emb @ image_emb[ref_images].T      # (n_captions, 2000)
image_vs_ref = image_emb @ caption_emb[ref_captions].T      # (n_images, 2000)

sim = cosine_rows(image_emb[row_image], caption_emb[row_caption])
caption_ref_rows = caption_vs_ref[row_caption]

clip_features = pd.DataFrame({
    "row_id": all_rows["row_id"],
    "clip_sim": sim,
    "clip_sim_txt_norm": sim - caption_ref_rows.mean(axis=1),
    "clip_sim_img_norm": sim - image_vs_ref[row_image].mean(axis=1),
    "clip_sim_txt_rank": (caption_ref_rows < sim[:, None]).mean(axis=1),
})
clip_features.describe().round(3)

In [ ]:
train_labels = data["train_sample"][["row_id", "image_fake", "text_fake", "image_manipulation", "text_manipulation"]]
clip_check = train_labels.merge(clip_features, on="row_id")

clip_auc = pd.concat(
    [auc_by_manipulation(clip_check, c) for c in clip_features.columns if c != "row_id"], axis=1
).T.round(3)
clip_auc

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
for ax, col in zip(axes, ["clip_sim", "clip_sim_txt_norm"]):
    sns.kdeplot(data=clip_check, x=col, hue="text_manipulation", hue_order=["none", "text_attribute", "text_swap"],
                common_norm=False, fill=True, alpha=0.3, ax=ax)
    ax.set_title(f"{col} by caption manipulation (train_sample)")
plt.tight_layout()
plt.show()

Swapped captions fit their photo clearly worse than genuine ones, which a text-only model cannot see. `text_attribute` shifts similarity only slightly, since flipping one or two sentiment words barely changes what the caption describes; that manipulation is left to the text features (sentiment, fluency).

In [ ]:
save_features(clip_features, "clip")

In [ ]:
# Raw embeddings for models that learn from them directly (float16 halves the file size; precision loss is negligible)
dim = image_emb.shape[1]
emb_features = pd.concat([
    all_rows[["row_id"]],
    pd.DataFrame(image_emb[row_image].astype(np.float16), columns=[f"clip_img_{i:03d}" for i in range(dim)]),
    pd.DataFrame(caption_emb[row_caption].astype(np.float16), columns=[f"clip_txt_{i:03d}" for i in range(dim)]),
], axis=1)
save_features(emb_features, "clip_emb")
del emb_features

## 3. Face-Caption Similarity

Each detected face is cropped (with a 20% margin) and embedded. CLIP knows many public figures, so a swapped face should match a caption naming a specific person less well than the real face did. As in `03`, the edited face is not always the largest, so similarity is summarised across all faces in the image.

In [ ]:
face_owner = np.repeat(np.arange(len(images)), images["mtcnn_boxes"].str.len().to_numpy())
face_paths = images["image"].to_numpy()[face_owner]
face_boxes = [box.tolist() for boxes in images["mtcnn_boxes"] for box in boxes]
face_area = np.array([(b[2] - b[0]) * (b[3] - b[1]) for b in face_boxes])
print(f"{len(face_boxes):,} face crops")

face_emb = cached("faces", lambda: encode_images(model, ImageDataset(face_paths, face_boxes), device,
                                                 num_workers=NUM_WORKERS))

In [ ]:
# One row per (dataset row, face): pair every face of the row's image with the row's caption
faces_per_image = np.bincount(face_owner, minlength=len(images))
first_face = np.concatenate([[0], np.cumsum(faces_per_image)[:-1]])
n_faces_row = faces_per_image[row_image]
row_of_pair = np.repeat(np.arange(len(all_rows)), n_faces_row)
face_of_pair = np.concatenate([first_face[i] + np.arange(n) for i, n in zip(row_image, n_faces_row)])

pairs = pd.DataFrame({
    "row": row_of_pair,
    "sim": cosine_rows(face_emb[face_of_pair], caption_emb[row_caption[row_of_pair]]),
    "area": face_area[face_of_pair],
})
grouped = pairs.groupby("row")["sim"]
largest = pairs.loc[pairs.groupby("row")["area"].idxmax()].set_index("row")["sim"]

face_features = pd.DataFrame({
    "row_id": all_rows["row_id"],
    "clip_face_sim_max": grouped.max(),
    "clip_face_sim_min": grouped.min(),
    "clip_face_sim_mean": grouped.mean(),
    "clip_face_sim_largest": largest,
    # Does the best-matching face fit the caption better than the whole image does?
    "clip_face_minus_image": grouped.max() - clip_features["clip_sim"],
})
face_features.describe().round(3)

In [ ]:
face_check = train_labels.merge(face_features, on="row_id")
face_auc = pd.concat(
    [auc_by_manipulation(face_check, c) for c in face_features.columns if c != "row_id"], axis=1
).T.round(3)
face_auc

In [ ]:
save_features(face_features, "clip_face")

## 4. Near-Duplicate Images Across Splits

`02_data_cleaning` found 38 captions shared across splits, suggesting some photos were scraped twice under different article IDs. CLIP embeddings can find duplicate photos even when the captions differ. For each validation and test original, we find its most similar `train_sample` original (`train_sample` covers about a third of train, so this undercounts).

In [ ]:
originals = all_rows["fake_cls"].eq("orig").to_numpy()
train_orig = np.unique(row_image[originals & train_mask])

dup_rows = []
for split in ["validation", "test"]:
    split_orig = np.unique(row_image[originals & all_rows["row_id"].str.startswith(split + "_").to_numpy()])
    best = np.concatenate([(image_emb[split_orig[i:i + 2048]] @ image_emb[train_orig].T).max(axis=1)
                           for i in range(0, len(split_orig), 2048)])
    for threshold in [0.95, 0.98]:
        dup_rows.append({"split": split, "threshold": threshold, "originals": len(split_orig),
                         "near_duplicates_in_train_sample": int((best >= threshold).sum()),
                         "share_%": round(100 * (best >= threshold).mean(), 2)})
pd.DataFrame(dup_rows)

# Summary

Fill in after running.